# Caso de Estudio: Fase de Procesamiento y Limpieza de Datos (Process)
## Proyecto: Plantillas Oficiales del Mundial 2026

En este cuaderno tomaremos el archivo de datos crudos generado vía Web Scraping (`datos_crudos_mundial.csv`) y aplicaremos transformaciones avanzadas con **Pandas** para corregir problemas de calidad de datos, remover texto residual de Wikipedia y estandarizar las columnas para su posterior análisis.

### Objetivos de Limpieza:
1. **Columna Nombre:** Remover anotaciones como `(captain)` o símbolos de capitanía.
2. **Columna Edad_Fecha:** Separar la fecha de nacimiento (en formato `YYYY-MM-DD`) y calcular/extraer la edad como un valor numérico entero limpio.
3. **Columna Club:** Estandarizar espacios en blanco y caracteres especiales ocultos.
4. **Estructura Final:** Guardar un dataset maestro limpio (`plantillas_limpias_mundial2026.csv`).

In [1]:
import pandas as pd
import re # Librería de Expresiones Regulares para limpiar textos complejos

# Cargar el dataset crudo
df = pd.read_csv("datos_crudos_mundial.csv")

# Ver las dimensiones iniciales y los tipos de datos
print(f"Dataset cargado con {df.shape[0]} filas y {df.shape[1]} columnas.\n")
df.head()

Dataset cargado con 1248 filas y 6 columnas.



,Tabla_Index,Numero,Posicion,Nombre,Edad_Fecha,Club
0,0,1,1GK,Matěj Kovář,"(2000-05-17)May 17, 2000 (aged 26)",PSV Eindhoven
1,0,2,2DF,David Zima,"(2000-11-08)November 8, 2000 (aged 25)",Slavia Prague
2,0,3,2DF,Tomáš Holeš,"(1993-03-31)March 31, 1993 (aged 33)",Slavia Prague
3,0,4,2DF,Robin Hranáč,"(2000-01-29)January 29, 2000 (aged 26)",TSG Hoffenheim
4,0,5,2DF,Vladimír Coufal,"(1992-08-22)August 22, 1992 (aged 33)",TSG Hoffenheim


### Paso 1: Limpieza de la columna 'Nombre'
Muchos jugadores vienen con la etiqueta `(captain)` adjunta a su nombre. Utilizaremos el método `.str.replace()` junto con expresiones regulares básicas para eliminar este texto y dejar únicamente el nombre limpio del futbolista.

In [2]:
# Aplicar limpieza para remover "(captain)" sin importar mayúsculas/minúsculas y espacios extras
df['Nombre'] = df['Nombre'].str.replace(r'\s*\(captain\)\s*', '', regex=True, flags=re.IGNORECASE)

# Verificar que los nombres de los capitanes hayan quedado limpios
df.head()

,Tabla_Index,Numero,Posicion,Nombre,Edad_Fecha,Club
0,0,1,1GK,Matěj Kovář,"(2000-05-17)May 17, 2000 (aged 26)",PSV Eindhoven
1,0,2,2DF,David Zima,"(2000-11-08)November 8, 2000 (aged 25)",Slavia Prague
2,0,3,2DF,Tomáš Holeš,"(1993-03-31)March 31, 1993 (aged 33)",Slavia Prague
3,0,4,2DF,Robin Hranáč,"(2000-01-29)January 29, 2000 (aged 26)",TSG Hoffenheim
4,0,5,2DF,Vladimír Coufal,"(1992-08-22)August 22, 1992 (aged 33)",TSG Hoffenheim


### Paso 2: El Desafío de la Columna 'Edad_Fecha'
La columna `Edad_Fecha` extraída de Wikipedia tiene una estructura compleja. Por ejemplo: `(2000-05-17)May 17, 2000 (aged 26)`.
Necesitamos extraer de ahí:
1. La fecha de nacimiento estándar en formato ISO `YYYY-MM-DD` (ej: `2000-05-17`).
2. La edad exacta como un número entero (ej: `26`).

Para lograrlo de forma ultra eficiente, usaremos expresiones regulares (`Regex`) con Pandas.

In [3]:
# 1. Extraer la fecha de nacimiento que cumple con el patrón YYYY-MM-DD (4 dígitos - 2 dígitos - 2 dígitos)
df['Fecha_Nacimiento'] = df['Edad_Fecha'].str.extract(r'(\d{4}-\d{2}-\d{2})')

# 2. Extraer la edad que viene después del texto 'aged ' dentro de los paréntesis finales
df['Edad'] = df['Edad_Fecha'].str.extract(r'aged\s+(\d+)')

# 3. Convertir la columna Edad a tipo numérico entero (Int64 por si hay algún nulo flotando)
df['Edad'] = pd.to_numeric(df['Edad'], errors='coerce')

# 4. Eliminar la columna original combinada que ya no necesitamos
df = df.drop(columns=['Edad_Fecha'])

# Ver el resultado de la transformación
df.head()

,Tabla_Index,Numero,Posicion,Nombre,Club,Fecha_Nacimiento,Edad
0,0,1,1GK,Matěj Kovář,PSV Eindhoven,2000-05-17,26
1,0,2,2DF,David Zima,Slavia Prague,2000-11-08,25
2,0,3,2DF,Tomáš Holeš,Slavia Prague,1993-03-31,33
3,0,4,2DF,Robin Hranáč,TSG Hoffenheim,2000-01-29,26
4,0,5,2DF,Vladimír Coufal,TSG Hoffenheim,1992-08-22,33


### Paso 3: Limpieza y Estandarización de Posiciones
La columna `Posicion` contiene números prefijados (ej: `1GK`, `2DF`). Utilizaremos expresiones regulares para eliminar cualquier dígito al inicio del texto y dejar únicamente las siglas estándar de las posiciones del fútbol (GK, DF, MF, FW).

In [4]:
# Eliminar todos los números que estén al comienzo de la cadena en la columna Posicion
df['Posicion'] = df['Posicion'].str.replace(r'^\d+', '', regex=True)

# Verificar cómo quedaron las posiciones ahora
df.head()

,Tabla_Index,Numero,Posicion,Nombre,Club,Fecha_Nacimiento,Edad
0,0,1,GK,Matěj Kovář,PSV Eindhoven,2000-05-17,26
1,0,2,DF,David Zima,Slavia Prague,2000-11-08,25
2,0,3,DF,Tomáš Holeš,Slavia Prague,1993-03-31,33
3,0,4,DF,Robin Hranáč,TSG Hoffenheim,2000-01-29,26
4,0,5,DF,Vladimír Coufal,TSG Hoffenheim,1992-08-22,33


### Paso 4: Almacenamiento del Dataset Limpio (Fase de Compartir / Share)
Con todas las columnas críticas transformadas, convertidas a sus tipos de datos correctos y estandarizadas, procedemos a exportar el DataFrame final a un archivo CSV estructurado. Este archivo será la base para nuestras consultas SQL avanzadas y la posterior creación del tablero en Power BI/Tableau.

In [5]:
# Guardar el dataset limpio definitivo
df.to_csv("plantillas_limpias_mundial2026.csv", index=False)

print("¡Proceso de procesamiento y limpieza completado con éxito!")
print(f"Archivo final guardado como 'plantillas_limpias_mundial2026.csv' con {df.shape[0]} registros listos para el análisis.")

¡Proceso de procesamiento y limpieza completado con éxito!
Archivo final guardado como 'plantillas_limpias_mundial2026.csv' con 1248 registros listos para el análisis.
